In [1]:
import os
import pandas as pd
import numpy as np
from scipy.spatial import distance
import scipy.stats as stats 

import plotly.express as px
import matplotlib.patches as mpatches



from joblib import load

In [2]:
def build_wl_matrix(df_group, mouse_ids):
    """
    df_group: subset of rows for one video × box × date.
    mouse_ids: list of all mice you expect in the box (e.g. ["mouseRED","mouseBLUE","mouseGREEN","mouseYELLOW"]).
    Returns:
        wl_mat: n×n matrix where wl_mat[i,j] = wins of i over j
        idx_to_mouse: index→mouse dict (row/column order)
    """
    # initialize
    wl_mat = pd.DataFrame(
        0,
        index=mouse_ids,
        columns=mouse_ids,
        dtype=float
    )

    for _, row in df_group.iterrows():
        winner = row["chaser"]
        loser  = row["chased"]
        if winner in wl_mat.index and loser in wl_mat.columns and winner != loser:
            wl_mat.loc[winner, loser] += 1
    
    return wl_mat.values, {i: m for i, m in enumerate(mouse_ids)}
    
    

In [3]:
def davids_score_from_matrix(wl_mat):
    """
    David's score: guaranteed normDS ∈ [0, N-1]
    """
    n = wl_mat.shape[0]
    
    # Total wins for simple ranking fallback
    total_wins = wl_mat.sum()
    
    if total_wins < 4:  # Too sparse: simple win count ranking
        DS = wl_mat.sum(axis=1)
    else:
        # Full David's score
        total = wl_mat + wl_mat.T
        nonzero = total > 0
        
        P = np.zeros_like(wl_mat, dtype=float)
        P[nonzero] = wl_mat[nonzero] / total[nonzero]
        
        # Symmetric handling: undecided pairs get 0.5 exactly
        P[~nonzero] = 0.5
        
        w1 = P.sum(axis=1)
        l1 = (1 - P).sum(axis=1)
        w2 = P @ w1
        l2 = (1 - P) @ l1
        
        DS = w1 + w2 - l1 - l2
    
    # Clip to exact theoretical bounds
    normDS = np.clip((DS + n * (n - 1) / 2.0) / n, 0, n - 1)
    
    return DS, normDS


In [9]:
from pathlib import Path

def hierarchies_long_format_dirs(chase_dirs, mouse_labels):
    base_paths = []
    all_df = []
    
    for chase_dir in chase_dirs:
        #csv_path = chase_dir.rstrip('\\') + '\\chases_all.csv'
        
        csv_path = str(Path(chase_dir) / "chases_all.csv")
        base_path = csv_path.rsplit('chases_all.csv', 1)[0]
        
        #df = pd.read_csv(csv_path)
        df = pd.read_csv(csv_path,sep=";")
        #print(df)

        all_df.append(df)
        base_paths.append(base_path)
        print(f"✓ {base_path}")
    
    df_all = pd.concat(all_df, ignore_index=True)
    df_all = assign_behavioral_day_robust(df_all, group_cols=('box', 'group'))
    print(df_all)
    df_all = df_all.dropna(subset=['day'])
    print(df_all,"after df all")
    results = []
    rank_names = ['Alpha', 'Beta', 'Gamma', 'Delta']

    print("Shape:", df_all.shape)
    print(df_all.head())


    
    df_all.loc[df_all["day"].between(1,3),"timepoint"] = "baseline"
    df_all.loc[df_all["day"].between(4,6),"timepoint"] = "after stress"
    df_all.loc[df_all["day"].between(7,13),"timepoint"] = "after treatment"
    
    for (box, day,group), df_group in df_all.groupby(['box', 'day',"group"]):
        rep_row = df_group.iloc[0]
        
        wl_mat, idx_to_mouse = build_wl_matrix(df_group, list(mouse_labels))
        DS, normDS = davids_score_from_matrix(wl_mat)
        
        order = np.argsort(-normDS)
        for rank_pos, mouse_idx in enumerate(order):
            results.append({
                'video': rep_row.get('video', 'N/A'),
                'box': box,
                'mouse': idx_to_mouse[mouse_idx].replace('mouse', ''),
                
                'day': day,
                
                'date': rep_row.get('date', 'N/A'),
                "group": group,
                "timepoint": rep_row["timepoint"],
                
                'rank': rank_names[rank_pos],
                'normDS': normDS[mouse_idx]
            })
    
    df_long = pd.DataFrame(results).sort_values(["group",'box', 'day', 'rank'])
    return df_long, base_paths


In [17]:
def hierarchies_long_format_dirs_cum(chase_dirs, mouse_labels):
    base_paths = []
    all_df = []
    
    for chase_dir in chase_dirs:
        #csv_path = chase_dir.rstrip('\\') + '\\chases_all.csv'
        csv_path = str(Path(chase_dir) / "chases_all.csv")
        base_path = csv_path.rsplit('chases_all.csv', 1)[0]
        
        #df = pd.read_csv(csv_path)
        df = pd.read_csv(csv_path,sep=";")
        

        all_df.append(df)
        base_paths.append(base_path)
        print(f"✓ {base_path}")
    
    df_all = pd.concat(all_df, ignore_index=True)
    df_all = assign_behavioral_day_robust(df_all, group_cols=('box',"group"))
    df_all = df_all.dropna(subset=['day'])
    
    results = []
    rank_names = ['Alpha', 'Beta', 'Gamma', 'Delta']

    
    df_all.loc[df_all["day"].between(1,3),"timepoint"] = "baseline"
    df_all.loc[df_all["day"].between(4,6),"timepoint"] = "after stress"
    df_all.loc[df_all["day"].between(7,13),"timepoint"] = "after treatment"

    
    df_all.loc[df_all["day"].between(1,3),"hier_day"] = "1_2_3"
    df_all.loc[df_all["day"].between(4,6),"hier_day"] = "4_5_6"
    df_all.loc[df_all["day"].between(7,8),"hier_day"] = "7_8"
    df_all.loc[df_all["day"].between(9,10),"hier_day"] = "9_10"
    df_all.loc[df_all["day"].between(11,13),"hier_day"] = "11_12_13"

    #print(df_all["hier_day"])
    
    for (box, hier_day, group), df_group in df_all.groupby(['box', 'hier_day',"group"]):
        rep_row = df_group.iloc[0]
        
        wl_mat, idx_to_mouse = build_wl_matrix(df_group, list(mouse_labels))
        DS, normDS = davids_score_from_matrix(wl_mat)
        
        order = np.argsort(-normDS)
        for rank_pos, mouse_idx in enumerate(order):
            results.append({
                'video': rep_row.get('video', 'N/A'),
                'box': box,
                'mouse': idx_to_mouse[mouse_idx].replace('mouse', ''),
                
                'day': rep_row["day"],
                
                'date': rep_row.get('date', 'N/A'),
                "group": group,
                "timepoint": rep_row["timepoint"],
                
                'rank_cum': rank_names[rank_pos],
                'normDS_cum': normDS[mouse_idx]
            })
    
    df_long = pd.DataFrame(results).sort_values(["group",'box', 'day', 'rank_cum'])
    return df_long, base_paths

In [15]:
def assign_behavioral_day_robust(df, group_cols):
    """
    Robust behavioral day assignment:
    - For each box, sort all rows by date/timebin.
    - Group into behavioral nights: [19–23 of date D + 0–6 of date D+1]
    - Assign sequential day numbers (1,2,3,...) to each night block,
      even if timebin 19 is missing.
    """
    df = df.copy()
    #df['date_dt'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    print(df.columns.tolist())


    df['date_dt'] = pd.to_datetime(
    df['date'],
    format='%Y/%m/%d'
    #errors='coerce'
)


    
    def per_group(g):
        g = g.sort_values(['date_dt', 'timebin']).copy()
        
        # Create a "night anchor" for each row
        g['night_anchor'] = g['date_dt'].copy()
        
        # Rows in 0–6 belong to PREVIOUS night's behavioral day
        early_mask = (g['timebin'] >= 0) & (g['timebin'] <= 6)
        g.loc[early_mask, 'night_anchor'] -= pd.Timedelta(days=1)
        
        # Now group by night_anchor → behavioral day
        g['day'] = g.groupby('night_anchor').ngroup() + 1
        
        return g
    
    df = df.groupby(list(group_cols), group_keys=False).apply(per_group)
    
    return df


In [22]:
#RUN ONE AT A TIME

csv_paths = [
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\psilo32_1f-Sere-2025-03-14\videos\ctrl baseline\active phase\chases",
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\psilo32_1f-Sere-2025-03-14\videos\ctrl after stress\active phase\chases",
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\psilo32_1f-Sere-2025-03-14\videos\ctrl after treatment\active phase\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\psilo32_1f-Sere-2025-03-14\videos\ucms after stress\active phase\chases",
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\psilo32_1f-Sere-2025-03-14\videos\ucms baseline\active phase\chases",
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\psilo32_1f-Sere-2025-03-14\videos\ucms after treatment\active phase\chases",

    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\psilo32_2f-Sere-2025-04-11\videos\ctrl after treatment_2f\active phase\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\psilo32_2f-Sere-2025-04-11\videos\ctrl before stress_2f\active phase\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\psilo32_2f-Sere-2025-04-11\videos\ctrl_before_treatment_2f\active phase\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\psilo32_2f-Sere-2025-04-11\videos\ucms before stress_2f\active phase\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\psilo32_2f-Sere-2025-04-11\videos\ucms_after_treatment_2f\active phase\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\psilo32_2f-Sere-2025-04-11\videos\ucms_before_treatment_2f\active phase\chases"

    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Males\Psilo32_1m\ctrl_before_stress_1m_cropped\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Males\Psilo32_1m\ucms_before_stress_1m_cropped\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Males\Psilo32_2m\ctrl_before_stress_2m_cropped\chases"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Males\Psilo32_2m\ucms_before_stress_2m_cropped\chases"

    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Males\Psilo32_1m\hierarchies"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Males\Psilo32_2m\hierarchies"
    #r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_1f\hierarchies"
    r"L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\hierarchies"


    ]

df_hier,base_paths  = hierarchies_long_format_dirs(
    csv_paths,
    mouse_labels=("mouseBLUE","mouseRED", "mouseGREEN", "mouseYELLOW")
)


#LIKE THIS ONLY FOR AFTER TREATMENT

df_hier_cum,base_paths  = hierarchies_long_format_dirs_cum(
    csv_paths,
    mouse_labels=("mouseBLUE","mouseRED", "mouseGREEN", "mouseYELLOW")
)


✓ L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\hierarchies\
['video', 'pair', 'start', 'end', 'duration', 'chaser', 'chased', 'box', 'phase', 'timebin', 'date', 'timestamp', 'group']
                                video pair  start    end  duration  \
3209   2025-01-20_19-03-17_crop_green   RB  27735  27754        20   
3210   2025-01-20_19-03-17_crop_green   RG  22565  22609        45   
3211   2025-01-20_19-03-17_crop_green   RG  22620  22659        40   
3212   2025-01-20_19-03-17_crop_green   RG  22735  22754        20   
3213   2025-01-20_19-03-17_crop_green   RG  22765  22784        20   
...                               ...  ...    ...    ...       ...   
10582    2025-02-25_05-04-56_crop_red   BY  83490  83509        20   
9659     2025-02-25_06-04-56_crop_red   RB  27130  27154        25   
9660     2025-02-25_06-04-56_crop_red   RB  27195  27219        25   
10890    2025-02-25_06-04-56_crop_red   GY  33500  33534        35   
10891    2025-02-25

In [23]:
# Save hierarchy results to same directories
for i, base_path in enumerate(base_paths):
    print(base_path)
    output_path = base_path + 'hierarchy_results.csv'
    print(f"Saved: {output_path}")

# Or save ALL results with phase identifier
df_hier.to_csv(base_paths[0] + 'hierarchies_alldays.csv', index=False)
df_hier_cum.to_csv(base_paths[0] + 'hierarchies_cumulative.csv', index=False)

L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\hierarchies\
Saved: L:\Lopez Laboratory - NEURO\Serena\PSILO2025\Behavior Females\Psilo32_2f\hierarchies\hierarchy_results.csv
